# Relacionando as bases

## 1 Montando as bases de cada arquivo

In [0]:
import pyspark.sql.functions as f
from itertools import chain

files_path = "/Volumes/academy/dados-streaing-filmes"

def to_date_multi_format(
    coluna, 
    formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")
):
    return f.coalesce(*[f.try_to_date(coluna, fmt) for fmt in formatos_suportados])

### 1.1 Base de Logs

In [0]:
df_logs = spark.read.csv(f"{files_path}/logs_streaming.csv", header=True, inferSchema=True)
print((df_logs.count(), len(df_logs.columns)))

df_logs = df_logs.dropDuplicates(subset=['session_id', 'user_id', 'movie_id'])

df_logs = (df_logs
           .withColumn('watch_time_minutes', f.expr('try_cast(watch_time_minutes as float)').cast('int'))
           .withColumn('watch_time_minutes', f.when(f.col("watch_time_minutes") > 0, f.col("watch_time_minutes")).otherwise(0)))

df_logs = df_logs.withColumn("watch_time_hours", f.round(f.col("watch_time_minutes")/60, 2))

df_logs = (df_logs
           .withColumn("watch_category", f.when((f.col("watch_time_minutes") >= 120) & (df_logs.watch_time_minutes.isNotNull()), "Longo")
                       .otherwise(f.when(f.col("watch_time_minutes") < 120, "Curto").otherwise(None))))

mapeamento_status = {"complete": "completed", "erro": "error"}
mapa_expr_status  = f.create_map([f.lit(x) for x in chain(*mapeamento_status.items())])
df_logs           = (df_logs
                     .withColumn("playback_status", f.lower(f.trim(f.col("playback_status"))))
                     .withColumn("playback_status", f.when(f.col("playback_status").isin(list(mapeamento_status.keys())), mapa_expr_status[f.col("playback_status")])
                                 .otherwise(f.col("playback_status"))))

df_logs = df_logs.withColumn( "subscription_type", f.lower("subscription_type") )

mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])
df_logs = (df_logs
           .withColumn("country", f.lower(f.trim(f.col("country"))))
           .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country"))))

df_logs = (df_logs
           .withColumn("device", f.lower(f.trim(f.col("device"))))
           .withColumn('device', f.regexp_replace('device', '_', ' ')))

df_logs = (df_logs
           .withColumn('rating', f.expr('try_cast(rating as float)').cast('int'))
           .withColumn('rating', f.when(f.col("rating") > 5, 5).otherwise(f.col("rating"))))

# def to_date_multi_format(coluna, formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS")):
#     return f.coalesce(*[f.try_to_date(coluna, fmt) for fmt in formatos_suportados])
df_logs = df_logs.withColumn("watch_date", to_date_multi_format(f.split(df_logs["watch_date"], ' ')[0]))

df_logs = df_logs.dropna(subset=['user_id']).dropna(subset=['movie_id']).dropna(subset=['session_id'])

df_logs = df_logs.fillna({"country": "desconhecido", "subscription_type": "unknown"})

df_logs = df_logs.fillna("sem informacao")
print((df_logs.count(), len(df_logs.columns)))

(1007, 10)
(997, 12)


In [0]:
df_logs.printSchema()

root
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- movie_id: string (nullable = false)
 |-- device: string (nullable = false)
 |-- country: string (nullable = false)
 |-- subscription_type: string (nullable = false)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: date (nullable = true)
 |-- rating: integer (nullable = true)
 |-- playback_status: string (nullable = false)
 |-- watch_time_hours: double (nullable = true)
 |-- watch_category: string (nullable = false)



### 1.2 Base de Filmes

In [0]:
from datetime import datetime

df_filmes = spark.read.csv(f"{files_path}/catalogo_filmes.csv", header=True, inferSchema=True)
print((df_filmes.count(), len(df_filmes.columns)))

df_filmes = df_filmes.dropDuplicates(subset=['movie_id'])

map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
map_expr_genre = f.create_map([f.lit(x) for x in chain(*map_genre.items())])
df_filmes      = (df_filmes
                  .withColumn("genre", f.lower(f.trim(f.col("genre"))))
                  .withColumn("genre", f.when(f.col("genre").isin(list(map_genre.keys())), map_expr_genre[f.col("genre")]).otherwise(f.col("genre"))))

current_year = int(datetime.now().year)
df_filmes    = (df_filmes
                .withColumn('release_year', f.expr('try_cast(release_year as int)'))
                .withColumn('release_year', f.when(f.col("release_year") <= current_year, f.col("release_year")).otherwise(None)))

df_filmes = (df_filmes.withColumn('duration_minutes', f.expr('try_cast(duration_minutes as int)')))

df_filmes = (df_filmes.withColumn("content_type", f.lower(f.trim(f.col("content_type")))))

map_lang      = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
map_expr_lang = f.create_map([f.lit(x) for x in chain(*map_lang.items())])
df_filmes     = (df_filmes
                 .withColumn(
                     "original_language", 
                     f.when(f.col("original_language").isin(list(map_lang.keys())), map_expr_lang[f.col("original_language")])
                     .otherwise(f.col("original_language"))))

df_filmes = (df_filmes
             .withColumn("title", f.lower(f.trim(f.col("title"))))
             .withColumn("title", f.split(df_filmes["title"], r"\s\d+$")[0]))

df_filmes = df_filmes.fillna("sem informacao")
print((df_filmes.count(), len(df_filmes.columns)))

(62, 7)
(61, 7)


In [0]:
df_filmes.printSchema()

root
 |-- movie_id: string (nullable = false)
 |-- title: string (nullable = false)
 |-- genre: string (nullable = false)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- content_type: string (nullable = false)
 |-- original_language: string (nullable = false)



### 1.3 Base de Usuários

In [0]:
df_usuarios = spark.read.csv(f"{files_path}/usuarios.csv", header=True, inferSchema=True)
print((df_usuarios.count(), len(df_usuarios.columns)))

df_usuarios = df_usuarios.dropDuplicates(subset=['user_id'])

df_usuarios = (df_usuarios
               .withColumn('age', f.expr('try_cast(age as int)'))
               .withColumn('age', f.when(f.col("age") > 0, f.col("age")).otherwise(None)))

mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])
df_usuarios        = (df_usuarios
                      .withColumn("country", f.lower(f.trim(f.col("country"))))
                      .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country"))))

df_usuarios = df_usuarios.withColumn("signup_date", to_date_multi_format(f.split(df_usuarios["signup_date"], ' ')[0]))

df_usuarios = df_usuarios.withColumn("subscription_type", f.lower(f.trim("subscription_type")))

print((df_usuarios.count(), len(df_usuarios.columns)))

(82, 6)
(81, 6)


In [0]:
df_usuarios.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)



## 2 Fazendo relacionamentos entre as bases

In [0]:
df_logs.limit(5).show()
df_filmes.limit(5).show()
df_usuarios.limit(5).show()

+-----------+-------+--------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+
| session_id|user_id|movie_id|        device|country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|
+-----------+-------+--------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+
|SD9E82F7AD2|  U0021|   M0010|      smart tv|espanha|            trial|               166|2026-04-23|     1|      buffering|            2.77|         Longo|
|S5A91FD1643|  U0026|   M0016|sem informacao|  chile|            basic|               120|2026-05-07|     0|          error|             2.0|         Longo|
|S04667B5896|  U0058|   M0006|        mobile| mexico|          premium|                97|2026-05-18|     3|      cancelled|            1.62|         Curto|
|S5429269E17|  U0010|   M0018|      smart tv| mexico|     

Ao observarmos as colunas dos dataframes, notamos que existem colunas com nomes comuns entre eles. E não só isso, mas o conteúdo dessas colunas parece ser o mesmo em ambas tabelas. Sendo assim, podemos usar essas colunas para relacionar bases umas as outras. Vamos começar fazendo um `left join` entre as bases de logs e filmes:

In [0]:
df_logs_filmes = df_logs.join(df_filmes, on='movie_id', how='left')

print("Shape dos dataframes:")
print(f"df_logs: ({df_logs.count()}, {len(df_logs.columns)})")
print(f"df_filmes: ({df_filmes.count()}, {len(df_filmes.columns)})")
print(f"df_logs_filmes: ({df_logs_filmes.count()}, {len(df_logs_filmes.columns)})\n")

df_logs_filmes.limit(5).show()

Shape dos dataframes:
df_logs: (997, 12)
df_filmes: (61, 7)
df_logs_filmes: (997, 18)

+--------+-----------+-------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+
|movie_id| session_id|user_id|        device|country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title|         genre|release_year|duration_minutes|  content_type|original_language|
+--------+-----------+-------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+
|   M0010|SD9E82F7AD2|  U0021|      smart tv|espanha|            trial|               166|2026-04-23|     1|      buffering|            2.77|         Longo| 

Como sabemos que a coluna `title` estava preenchida em todas as linhas da base de filmes, podemos utiliza-la para avaliar se todas as linhas retornadas possuem relacionamento:

In [0]:
# coluna 'title' esta preenchida em todas as linhas
df_filmes.filter(df_filmes['title'].isNull()).show()

+--------+-----+-----+------------+----------------+------------+-----------------+
|movie_id|title|genre|release_year|duration_minutes|content_type|original_language|
+--------+-----+-----+------------+----------------+------------+-----------------+
+--------+-----+-----+------------+----------------+------------+-----------------+



In [0]:
# temos relacionamento em todos linhas
df_logs_filmes.filter(df_logs_filmes['title'].isNull()).show()

+--------+----------+-------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+----------------+------------+-----------------+
|movie_id|session_id|user_id|device|country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|title|genre|release_year|duration_minutes|content_type|original_language|
+--------+----------+-------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+----------------+------------+-----------------+
+--------+----------+-------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+----------------+------------+-----------------+



In [0]:
df_final = df_logs_filmes.join(df_usuarios, on='user_id', how='left')

print("Shape dos dataframes:")
print(f"df_logs: ({df_logs.count()}, {len(df_logs.columns)})")
print(f"df_filmes: ({df_filmes.count()}, {len(df_filmes.columns)})")
print(f"df_usuarios: ({df_usuarios.count()}, {len(df_usuarios.columns)})")
print(f"df_logs_filmes: ({df_logs_filmes.count()}, {len(df_logs_filmes.columns)})")
print(f"df_final:\t  ({df_final.count()}, {len(df_final.columns)})\n")

df_final.limit(5).show()

Shape dos dataframes:
df_logs: (997, 12)
df_filmes: (61, 7)
df_usuarios: (81, 6)
df_logs_filmes: (997, 18)
df_final:	  (997, 23)

+-------+--------+-----------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|user_id|movie_id| session_id|        device|country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title|         genre|release_year|duration_minutes|  content_type|original_language|     user_name|age|  country|signup_date|subscription_type|
+-------+--------+-----------+--------------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+---------------

In [0]:
# temos relacionamento em todos linhas
df_final.filter(df_final['title'].isNull()).show()

+-------+--------+----------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+----------------+------------+-----------------+---------+---+-------+-----------+-----------------+
|user_id|movie_id|session_id|device|country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|title|genre|release_year|duration_minutes|content_type|original_language|user_name|age|country|signup_date|subscription_type|
+-------+--------+----------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+----------------+------------+-----------------+---------+---+-------+-----------+-----------------+
+-------+--------+----------+------+-------+-----------------+------------------+----------+------+---------------+----------------+--------------+-----+-----+------------+---------

In [0]:
df_final.printSchema()

root
 |-- user_id: string (nullable = false)
 |-- movie_id: string (nullable = false)
 |-- session_id: string (nullable = false)
 |-- device: string (nullable = false)
 |-- country: string (nullable = false)
 |-- subscription_type: string (nullable = false)
 |-- watch_time_minutes: integer (nullable = true)
 |-- watch_date: date (nullable = true)
 |-- rating: integer (nullable = true)
 |-- playback_status: string (nullable = false)
 |-- watch_time_hours: double (nullable = true)
 |-- watch_category: string (nullable = false)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- content_type: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)

